In [1]:
import os
import shutil
import tempfile
from typing import List

# --- LangChain Imports ---
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Vector DB Imports ---
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import LanceDB
import lancedb

# --- CONFIGURATION ---
# Choose your DB: "faiss", "chroma", or "lancedb"
VECTOR_DB_TYPE = "faiss" 
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Ensure API Key is set
os.environ["OPENAI_API_KEY"] = ""
# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = "sk-..."

# 1. LOAD AND CHUNK DOCUMENT
def load_and_split_document(file_path: str):
    """
    Loads a PDF or Text file and splits it into chunks.
    """
    print(f"📄 Loading document: {file_path}")
    
    if file_path.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.endswith(".txt"):
        loader = TextLoader(file_path)
    else:
        raise ValueError("Unsupported file format. Please use .pdf or .txt")

    docs = loader.load()
    
    # Split text
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    chunks = text_splitter.split_documents(docs)
    print(f"✂️  Split document into {len(chunks)} chunks.")
    return chunks

# 2. INITIALIZE VECTOR STORE (MODULAR)
def get_vector_store(chunks, db_type="faiss"):
    """
    Creates and returns a vector store based on the selected db_type.
    """
    embeddings = OpenAIEmbeddings()
    print(f"💾 Creating {db_type.upper()} Vector Store...")

    if db_type == "faiss":
        print("\ncreating FAISS vector")
        # FAISS is purely in-memory
        vectorstore = FAISS.from_documents(chunks, embeddings)
        
    elif db_type == "chroma":
        # Chroma in-memory mode (ephemeral)
        vectorstore = Chroma.from_documents(
            documents=chunks, 
            embedding=embeddings,
            collection_name="rag_collection"
        )
        
    elif db_type == "lancedb":
        # LanceDB requires a local path, using temp dir for "in-memory" feel
        temp_dir = tempfile.mkdtemp()
        db = lancedb.connect(temp_dir)
        table = db.create_table(
            "rag_table",
            data=[
                {"vector": embeddings.embed_query("test"), "text": "test", "id": "0"}
            ],
            mode="overwrite"
        )
        vectorstore = LanceDB.from_documents(
            chunks, 
            embeddings, 
            connection=table
        )
    else:
        raise ValueError("Invalid Vector DB Type. Choose 'faiss', 'chroma', or 'lancedb'")

    return vectorstore

# 3. RAG PIPELINE GENERATION
def run_rag_pipeline(vectorstore, user_query: str):
    """
    Retrieves context and generates an answer using LLM.
    """
    print(f"🔍 Querying: '{user_query}'")
    
    # Create Retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    
    # Initialize LLM
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

    # Define Prompt
    template = """You are a helpful assistant. Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    
    Context:
    {context}

    Question: {question}
    
    Answer:"""
    
    prompt = ChatPromptTemplate.from_template(template)

    # Build Chain
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Execute
    response = rag_chain.invoke(user_query)
    return response

def run_manual_rag_pipeline(vectorstore, user_query: str):
    """
    Runs RAG manually: 
    1. Converts query to vector
    2. Searches DB with that vector
    3. Builds prompt manually
    4. Calls LLM
    """
    print(f"🔍 Querying: '{user_query}'")
    
    # --- STEP 1: Convert User Query to Vector ---
    # We need the same embedding model used to create the vector store
    embeddings = OpenAIEmbeddings()
    
    # This is the math part: String -> List of Floats
    query_vector = embeddings.embed_query(user_query)
    
    # OPTIONAL: Print the start of the vector to prove it exists
    print(f"🧮 Generated Vector (First 5 dims): {query_vector[:5]}...") 

    # --- STEP 2: Fetch Similar Docs using the Vector ---
    # We pass the vector (list of floats) directly to the DB
    docs = vectorstore.similarity_search_by_vector(query_vector, k=3)
    
    print(f"📚 Retrieved {len(docs)} relevant chunks.")
    for i, doc in enumerate(docs):
        print(f"   [Chunk {i+1}]: {doc.page_content[:50]}...")

    # --- STEP 3: Construct the Prompt Manually ---
    # Extract just the text content from the document objects
    context_text = "\n\n".join([d.page_content for d in docs])
    
    # Create the final string to send to the LLM
    final_prompt = f"""You are a helpful assistant.
    
    Context:
    {context_text}

    Question: {user_query}
    
    Answer:"""

    # --- STEP 4: Pass to LLM ---
    print("🤖 Sending to LLM...")
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    
    # invoke() takes a string or list of messages
    response = llm.invoke(final_prompt)
    
    return response.content

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # Create a dummy file for testing if you don't have one
    sample_file = "sample_data.txt"
    with open(sample_file, "w") as f:
        f.write("""
        LanceDB is an open-source database for vector-search built with persistent storage, 
        which greatly simplifies retrieval, filtering and management of embeddings.
        FAISS is a library for efficient similarity search and clustering of dense vectors.
        Chroma is the AI-native open-source embedding database.
        """)

    try:
        # 1. Load
        chunks = load_and_split_document(sample_file)
        
        # 2. Store (Change "faiss" to "chroma" or "lancedb" here)
        vector_db = get_vector_store(chunks, db_type=VECTOR_DB_TYPE)
        
        # 3. Query
        query = "What is the difference between FAISS and LanceDB?"
        answer = run_manual_rag_pipeline(vector_db, query)
        
        print("\n" + "="*30)
        print(f"🤖 AI Answer:\n{answer}")
        print("="*30)
        
    except Exception as e:
        print(f"Error: {e}")
    finally:
        # Cleanup dummy file
        if os.path.exists(sample_file):
            os.remove(sample_file)

📄 Loading document: sample_data.txt
✂️  Split document into 1 chunks.
💾 Creating FAISS Vector Store...

creating FAISS vector
🔍 Querying: 'What is the difference between FAISS and LanceDB?'
🧮 Generated Vector (First 5 dims): [0.006810775492340326, 0.03568344935774803, 0.007458426058292389, -0.004975765943527222, -0.004993175622075796]...
📚 Retrieved 1 relevant chunks.
   [Chunk 1]: LanceDB is an open-source database for vector-sear...
🤖 Sending to LLM...

🤖 AI Answer:
FAISS is a library for efficient similarity search and clustering of dense vectors, while LanceDB is an open-source database for vector-search built with persistent storage. FAISS focuses on similarity search and clustering, while LanceDB simplifies retrieval, filtering, and management of embeddings.


## !pip install faiss-cpu